In [248]:
import pandas as pd
import os
import re
import unicodedata
from datetime import datetime

In [249]:
# ============================================================
# CAMINHOS
# ============================================================

base_path = r'C:\Users\thiago.deoliveira\OneDrive - Secretaria da Educação do Estado de São Paulo\Contrato_Coleta de dados_Projeto_N1_N2-FDE-0A9MLZ-N'

arquivo_historico = os.path.join(base_path, 'Base_Historica_Avaliadores.xlsx')
arquivo_saida_atualizada = os.path.join(base_path, 'Base_Unificada_Lotes_Atualizada.xlsx')
arquivo_novos = os.path.join(base_path, 'Novos_Avaliadores.xlsx')

caminho_saida_liberacao = os.path.join(base_path, 'Liberacao_Acesso_App')
arquivo_liberacao = os.path.join(caminho_saida_liberacao, 'liberacao_acesso.xlsx')

arquivo_controle_full = os.path.join(base_path, 'controle_carga_full_liberacao.txt')


# ============================================================
# CONFIGURAÇÕES
# ============================================================

colunas_padrao = {
    'E-mail_Gestor_FDE': 'maria.tafner@fde.sp.gov.br',
    'Lotacao': 'FDE/DOS/GIO',
    'Projeto': 'MANUTENCAO_PREDIAL_N1',
    'Perfil_do_usuario': 'Avaliador_N1'
}

colunas_liberacao = [
    'Empresa',
    'Nome completo do avaliador',
    'RG',
    'CPF',
    'E-mail pessoal',
    'Telefone',
    'LOTE',
    'E-mail_Gestor_FDE',
    'Perfil_do_usuario',
    'Lotacao',
    'Projeto',
    'Cargo',
    'Número do Contrato FDE'
]


# ============================================================
# FUNÇÕES
# ============================================================

def normalizar(valor):
    if pd.isna(valor):
        return ''

    valor = str(valor).strip().lower()
    valor = unicodedata.normalize('NFKD', valor)

    valor = ''.join(
        c for c in valor
        if not unicodedata.combining(c)
    )

    valor = re.sub(r'\s+', ' ', valor)

    return valor


def aplicar_colunas_padrao(df):
    df = df.copy()

    for coluna, valor in colunas_padrao.items():
        df[coluna] = valor

    return df


def renomear_coluna_funcao_para_cargo(df):
    df = df.copy()

    mapa_renomeacao = {}

    for coluna in df.columns:
        coluna_normalizada = normalizar(coluna)

        if coluna_normalizada in ['funcao', 'função']:
            mapa_renomeacao[coluna] = 'Cargo'

    df = df.rename(columns=mapa_renomeacao)

    return df


def criar_chave(df):
    df = df.copy()

    colunas_chave = [
        'Nome completo do avaliador',
        'RG',
        'CPF',
        'E-mail pessoal',
        'Telefone',
        'Cargo'
    ]

    colunas_existentes = [
        col for col in colunas_chave
        if col in df.columns
    ]

    if not colunas_existentes:
        raise ValueError('Nenhuma coluna de chave foi encontrada na base.')

    df['_CHAVE_UNICA'] = (
        df[colunas_existentes]
        .fillna('')
        .apply(
            lambda linha: '|'.join(normalizar(v) for v in linha),
            axis=1
        )
    )

    return df


def padronizar_colunas(df):
    df = df.copy()

    df.columns = [
        unicodedata.normalize('NFKD', col)
        .encode('ASCII', 'ignore')
        .decode('utf-8')
        .upper()
        .replace(' ', '_')
        .replace('-', '_')
        for col in df.columns
    ]

    return df


# ============================================================
# LEITURA DOS LOTES
# ============================================================

dfs = []

for i in range(1, 5):

    if i == 2:
        nomes_arquivos = [
            f'Cadastro_Avaliador_Lote{i}.xlsx',
            'Cadastro Pré Teste_TUV Rheinland - Lote 2.xlsx'
        ]
    else:
        nomes_arquivos = [
            f'Cadastro_Avaliador_Lote{i}.xlsx'
        ]

    for nome_arquivo in nomes_arquivos:

        caminho_arquivo = os.path.join(
            base_path,
            f'Lote {i}',
            nome_arquivo
        )

        print('\nLendo arquivo:')
        print(caminho_arquivo)

        if not os.path.exists(caminho_arquivo):
            print('Arquivo não encontrado.')
            continue

        df_temp = pd.read_excel(
            caminho_arquivo,
            dtype=str
        )

        df_temp = renomear_coluna_funcao_para_cargo(df_temp)

        if i in [3, 4]:
            df_temp['LOTE'] = 'Lote 3/4'
        else:
            df_temp['LOTE'] = f'Lote {i}'

        df_temp = aplicar_colunas_padrao(df_temp)

        dfs.append(df_temp)


if not dfs:
    raise ValueError('Nenhum arquivo foi encontrado para processamento.')


# ============================================================
# BASE ATUAL CONSOLIDADA
# ============================================================

df_atual = pd.concat(
    dfs,
    ignore_index=True
)

df_atual = aplicar_colunas_padrao(df_atual)
df_atual = criar_chave(df_atual)

df_atual = (
    df_atual
    .drop_duplicates(subset=['_CHAVE_UNICA'], keep='first')
    .reset_index(drop=True)
)

df_atual['DATA_PROCESSAMENTO'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')


# ============================================================
# HISTÓRICO E NOVOS REGISTROS
# ============================================================

if os.path.exists(arquivo_historico):

    print('\nHistórico encontrado.')

    df_historico = pd.read_excel(
        arquivo_historico,
        dtype=str
    )

    df_historico = aplicar_colunas_padrao(df_historico)
    df_historico = renomear_coluna_funcao_para_cargo(df_historico)

    if '_CHAVE_UNICA' not in df_historico.columns:
        df_historico = criar_chave(df_historico)

    chaves_historico = set(
        df_historico['_CHAVE_UNICA'].fillna('')
    )

    df_novos = df_atual[
        ~df_atual['_CHAVE_UNICA'].isin(chaves_historico)
    ].copy()

    df_novos = aplicar_colunas_padrao(df_novos)

    df_historico_atualizado = pd.concat(
        [df_historico, df_novos],
        ignore_index=True
    )

    df_historico_atualizado = aplicar_colunas_padrao(df_historico_atualizado)

    df_historico_atualizado = (
        df_historico_atualizado
        .drop_duplicates(subset=['_CHAVE_UNICA'], keep='first')
        .reset_index(drop=True)
    )

else:

    print('\nHistórico não encontrado.')
    print('Criando nova base histórica.')

    df_novos = df_atual.copy()
    df_novos = aplicar_colunas_padrao(df_novos)

    df_historico_atualizado = df_atual.copy()
    df_historico_atualizado = aplicar_colunas_padrao(df_historico_atualizado)


# ============================================================
# SALVAR BASES DE CONTROLE
# ============================================================

df_novos.to_excel(arquivo_novos, index=False)
df_historico_atualizado.to_excel(arquivo_historico, index=False)
df_atual.to_excel(arquivo_saida_atualizada, index=False)


# ============================================================
# CARGA FULL / INCREMENTAL
# ============================================================

os.makedirs(caminho_saida_liberacao, exist_ok=True)

if not os.path.exists(arquivo_controle_full):

    df_base_liberacao = df_atual.copy()
    tipo_carga = 'CARGA FULL'

    with open(arquivo_controle_full, 'w', encoding='utf-8') as f:
        f.write(
            f'Carga full realizada em '
            f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
        )

else:

    df_base_liberacao = df_novos.copy()
    tipo_carga = 'CARGA INCREMENTAL'


# ============================================================
# BASE FINAL LIBERAÇÃO
# ============================================================

colunas_existentes_liberacao = [
    col for col in colunas_liberacao
    if col in df_base_liberacao.columns
]

df_final = df_base_liberacao[
    colunas_existentes_liberacao
].copy()

df_final = padronizar_colunas(df_final)


# ============================================================
# SALVAR LIBERAÇÃO
# ============================================================

df_final.to_excel(
    arquivo_liberacao,
    index=False
)


# ============================================================
# RESUMO
# ============================================================

print('\n======================================')
print('RESUMO DO PROCESSAMENTO')
print('======================================')
print(f'Tipo de carga: {tipo_carga}')
print(f'Total consolidado atual: {len(df_atual)}')
print(f'Novos registros encontrados: {len(df_novos)}')
print(f'Total histórico atualizado: {len(df_historico_atualizado)}')
print(f'Total salvo em liberacao_acesso.xlsx: {len(df_final)}')
print(f'Arquivo salvo em: {arquivo_liberacao}')
print('======================================')


Lendo arquivo:
C:\Users\thiago.deoliveira\OneDrive - Secretaria da Educação do Estado de São Paulo\Contrato_Coleta de dados_Projeto_N1_N2-FDE-0A9MLZ-N\Lote 1\Cadastro_Avaliador_Lote1.xlsx

Lendo arquivo:
C:\Users\thiago.deoliveira\OneDrive - Secretaria da Educação do Estado de São Paulo\Contrato_Coleta de dados_Projeto_N1_N2-FDE-0A9MLZ-N\Lote 2\Cadastro_Avaliador_Lote2.xlsx

Lendo arquivo:
C:\Users\thiago.deoliveira\OneDrive - Secretaria da Educação do Estado de São Paulo\Contrato_Coleta de dados_Projeto_N1_N2-FDE-0A9MLZ-N\Lote 2\Cadastro Pré Teste_TUV Rheinland - Lote 2.xlsx

Lendo arquivo:
C:\Users\thiago.deoliveira\OneDrive - Secretaria da Educação do Estado de São Paulo\Contrato_Coleta de dados_Projeto_N1_N2-FDE-0A9MLZ-N\Lote 3\Cadastro_Avaliador_Lote3.xlsx

Lendo arquivo:
C:\Users\thiago.deoliveira\OneDrive - Secretaria da Educação do Estado de São Paulo\Contrato_Coleta de dados_Projeto_N1_N2-FDE-0A9MLZ-N\Lote 4\Cadastro_Avaliador_Lote4.xlsx

Histórico não encontrado.
Criando nov